# RFM 백테스트

**사용 방법**
1. `셀 2`에서 본인 전략을 작성하세요.
2. 나머지 셀은 순서대로 실행하면 됩니다 (`Shift + Enter`).

자세한 내용은 `backtest/README.md`를 참고하세요.

## 셀 1 — 설치 및 임포트

처음 실행하는 경우 아래 주석을 해제하고 실행하세요.

In [ ]:
# !pip install yfinance pandas numpy scipy matplotlib pyarrow

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # timefolio-engine 루트를 경로에 추가

import pandas as pd
import matplotlib.pyplot as plt

from backtest.data    import load_universe, download_prices, get_sector_map, save_prices, load_prices
from backtest.engine  import run_backtest
from backtest.metrics import print_report

## ✏️ 셀 2 — 전략 작성 (여기만 수정하세요)

- **입력** `prices`: 신호 계산 기준일까지의 수정주가 (`DataFrame`)
- **출력**: 종목별 목표 비중 (`Series`, 종목코드 → 비중)
- 비중 합이 1.0이 아니어도 됩니다. 엔진이 자동으로 정규화합니다.
- 종목 한도(15%), 섹터 비중, 소형주 제약은 **자동 적용**됩니다.

> **룩어헤드 주의**: `prices`의 마지막 행이 신호 계산 기준일(t) 종가입니다.  
> 반환된 비중은 **t+1일부터** 적용됩니다.

In [ ]:
def my_strategy(prices: pd.DataFrame) -> pd.Series:
    """종목별 목표 비중 반환 (종목코드 → 비중)."""

    # ── 예시: 3개월 모멘텀 전략 ───────────────────────────────────────
    # 최근 63 거래일(≈ 3개월) 수익률 상위 20% 종목을 균등 비중으로 편입
    # 단기 반전 노이즈 제거를 위해 최근 5일은 제외
    lookback, skip = 63, 5

    if len(prices) < lookback + skip + 2:
        return pd.Series(dtype=float)

    momentum = (prices.iloc[-(skip + 1)] / prices.iloc[-(lookback + skip + 1)] - 1).dropna()

    n = max(10, min(30, int(len(momentum) * 0.2)))
    selected = momentum.nlargest(n).index

    return pd.Series(1.0 / n, index=selected)  # 균등 비중
    # ─────────────────────────────────────────────────────────────────

## 셀 3 — 기간 및 데이터 설정

In [ ]:
BACKTEST_START = "2022-01-01"
BACKTEST_END   = "2026-04-01"   # exclusive
PRICE_CACHE    = "prices_cache.parquet"  # 한 번 저장하면 재실행 시 빠릅니다

universe   = load_universe()
sector_map = get_sector_map(universe)
print(f"유니버스: {len(universe)}종목")

## 셀 4 — 주가 다운로드

처음 실행 시 수 분 걸립니다. 이후에는 캐시에서 자동 로드됩니다.

In [ ]:
if os.path.exists(PRICE_CACHE):
    prices = load_prices(PRICE_CACHE)
    print(f"캐시에서 로드: {PRICE_CACHE}")
else:
    print("yfinance 다운로드 중... (5~10분 소요)")
    prices = download_prices(universe, start=BACKTEST_START, end=BACKTEST_END)
    save_prices(prices, PRICE_CACHE)
    print(f"캐시 저장 완료: {PRICE_CACHE}")

print(f"기간: {prices.index[0].date()} ~ {prices.index[-1].date()}")
print(f"종목 수 (데이터 있는 것): {len(prices.columns)}개")
prices.tail(3)

## 셀 5 — 백테스트 실행

In [ ]:
result = run_backtest(
    prices      = prices,
    strategy_fn = my_strategy,
    sector_map  = sector_map,
    # is_large_cap = None,  # 시총 데이터 있으면 dict{yf_ticker: bool}로 전달
)

print_report(result["nav"], result["turnover_log"], result["turnover_violations"])

## 셀 6 — 수익률 차트

In [ ]:
nav = result["nav"]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), gridspec_kw={"height_ratios": [3, 1]})

# 누적 수익률
cum = nav / nav.iloc[0] * 100
cum.plot(ax=axes[0], color="steelblue", label="전략")
axes[0].set_title("포트폴리오 누적 수익률")
axes[0].set_ylabel("수익률 지수 (기준=100)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 낙폭 (Drawdown)
drawdown = (nav - nav.cummax()) / nav.cummax() * 100
drawdown.plot(ax=axes[1], color="salmon", label="낙폭")
axes[1].fill_between(drawdown.index, drawdown, 0, color="salmon", alpha=0.3)
axes[1].set_ylabel("낙폭 (%)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 셀 7 — 주간 회전율 체크

In [ ]:
tlog = result["turnover_log"]

if tlog.empty:
    print("회전율 데이터 없음")
else:
    fig, ax = plt.subplots(figsize=(13, 3))
    colors = ["salmon" if v else "steelblue" for v in tlog["violated"]]
    ax.bar(tlog["date"], tlog["turnover"] * 100, color=colors, width=3)
    ax.axhline(5, color="red", linestyle="--", linewidth=1, label="기준 5%")
    ax.set_title("주간 회전율  (빨간 막대 = 위반)")
    ax.set_ylabel("회전율 (%)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"위반 횟수: {result['turnover_violations']}회 / 허용 3회")

## 셀 8 — 마지막 날 포트폴리오 확인

In [ ]:
final = result["final_weights"].sort_values(ascending=False)
print(f"편입 종목 수: {len(final)}개")
final.head(20).rename("비중").map("{:.2%}".format).to_frame()